In [ ]:
import os
import csv
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tf_keras_vis.gradcam_plus_plus import GradcamPlusPlus
from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
from tf_keras_vis.utils.scores import CategoricalScore
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import cv2 
from tqdm import tqdm
from tensorflow.keras.layers import (
    Layer, Conv2D, Dense,
    GlobalAveragePooling2D, GlobalMaxPooling2D,
    Reshape, Multiply, Add, Activation, Concatenate
)

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom", name="F1Score")
class F1Score(tf.keras.metrics.Metric):
    """
    Custom Keras metric to compute the F1 Score.
    The F1 score is the harmonic mean of precision and recall.
    """

    def __init__(self, name='f1_score', **kwargs):
        """
        Initializes the F1Score metric. 
        - Uses Keras' Precision and Recall metrics as intermediate steps.
        """
        super().__init__(name=name, **kwargs)
        self.precision = tf.keras.metrics.Precision()  # Precision metric
        self.recall = tf.keras.metrics.Recall()  # Recall metric

    def update_state(self, y_true, y_pred, sample_weight=None):
        """
        Updates the state of the metric.
        - This method is called during training to update precision and recall.
        - The precision and recall states are updated based on true and predicted values.
        """
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)

    def result(self):
        """
        Computes and returns the F1 score.
        - F1 score is calculated as the harmonic mean of precision and recall.
        - Prevents division by zero by adding a small epsilon value to the denominator.
        """
        p = self.precision.result()  # Get precision value
        r = self.recall.result()  # Get recall value
        return 2 * (p * r) / (p + r + tf.keras.backend.epsilon())  # Calculate F1 score

    def reset_states(self):
        """
        Resets the states of the precision and recall metrics.
        - This is called at the beginning of each epoch.
        """
        self.precision.reset_states()  # Reset precision state
        self.recall.reset_states()  # Reset recall state


In [ ]:
# Channel Attention Block
@tf.keras.utils.register_keras_serializable(package="Custom", name="ChannelAttention")
class ChannelAttention(Layer):
    """
    Channel Attention Block (CA) that computes channel-wise attention.
    
    Args:
        reduction: The factor for reducing the number of channels in the intermediate layer.
    """
    def __init__(self, reduction=16, **kwargs):
        super(ChannelAttention, self).__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        """
        Build the dense layers for the Channel Attention mechanism.
        - `shared_dense_one`: reduces channel dimensions.
        - `shared_dense_two`: restores the original channel dimensions.
        """
        channel = input_shape[-1]  # Number of channels in the input tensor
        self.shared_dense_one = Dense(channel // self.reduction, activation='relu', kernel_initializer='he_normal', use_bias=True)
        self.shared_dense_two = Dense(channel, kernel_initializer='he_normal', use_bias=True)

    def call(self, inputs):
        """
        Apply the Channel Attention mechanism:
        - Global Average Pooling (avg_pool) and Global Max Pooling (max_pool)
        - Process each through a set of shared dense layers.
        - Combine both attentions and apply a sigmoid function to get the attention weights.
        """
        # Global average and max pooling
        avg_pool = GlobalAveragePooling2D()(inputs)
        max_pool = GlobalMaxPooling2D()(inputs)

        # Apply shared dense layers for both average and max pooled features
        avg_pool = self.shared_dense_one(avg_pool)
        avg_pool = self.shared_dense_two(avg_pool)

        max_pool = self.shared_dense_one(max_pool)
        max_pool = self.shared_dense_two(max_pool)

        # Combine both attention signals and apply sigmoid activation
        attention = Add()([avg_pool, max_pool])
        attention = Activation('sigmoid')(attention)

        # Reshape the attention to match the input dimensions and apply multiplication
        attention = Reshape((1, 1, -1))(attention)
        return Multiply()([inputs, attention])

# Spatial Attention Block
@tf.keras.utils.register_keras_serializable(package="Custom", name="SpatialAttention")
class SpatialAttention(Layer):
    """
    Spatial Attention Block (SA) that computes spatial attention across channels.
    """
    def __init__(self, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)
        # 2D convolution for spatial attention with a kernel size of 7x7 and sigmoid activation
        self.conv2d = Conv2D(filters=1, kernel_size=7, strides=1, padding='same', activation='sigmoid')

    def call(self, inputs):
        """
        Apply the Spatial Attention mechanism:
        - Apply both average and max pooling across the channel axis to extract spatial features.
        - Concatenate the two features and apply convolution to get spatial attention.
        """
        # Apply global average and max pooling along the channel axis
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)

        # Concatenate both pooled features
        concat = Concatenate(axis=-1)([avg_pool, max_pool])

        # Apply convolution to compute spatial attention map
        attention = self.conv2d(concat)

        # Multiply the attention map with the input to highlight important spatial features
        return Multiply()([inputs, attention])

# Full CBAM Block
def cbam_block(inputs, reduction=16):
    """
    CBAM (Convolutional Block Attention Module) that sequentially applies
    Channel Attention and Spatial Attention blocks.

    Args:
        inputs: Input tensor to the CBAM block.
        reduction: The factor for reducing the channel dimension in the Channel Attention block.
    
    Returns:
        The output tensor after applying both attention mechanisms.
    """
    # Apply Channel Attention followed by Spatial Attention
    x = ChannelAttention(reduction)(inputs)
    x = SpatialAttention()(x)
    return x


In [ ]:
def preprocess_image(image):
    # Convert image to uint8 if needed (required for CLAHE)
    if image.dtype != np.uint8:
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # Convert to LAB color space
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    # Apply CLAHE to the L-channel (lightness)
    clahe = cv2.createCLAHE(clipLimit=0.01, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])

    # Convert back to RGB
    image_clahe = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # Apply Min-Max scaling to [0, 1]
    image_clahe = image_clahe.astype(np.float32)
    image_clahe = (image_clahe - np.min(image_clahe)) / (np.ptp(image_clahe) + 1e-8)  # np.ptp = max - min

    return image_clahe

In [ ]:
def generate_gradcam_pp_dataset(model,
                                dataset_path,
                                labels,
                                target_layer_name,
                                output_folder,
                                csv_path,
                                samples_per_class=10,
                                include_misclassified=False,
                                image_size=(224, 224)):
    """
    Generate Grad-CAM++ overlays for selected images from a multi-class dataset.

    Parameters:
        model                 : Trained CNN model
        dataset_path          : Root dataset folder containing subfolders per class
        labels                : List of class labels
        target_layer_name     : Layer name for Grad-CAM++
        output_folder         : Output folder (will contain /original and /gradcam subfolders)
        csv_path              : CSV path to save predictions
        samples_per_class     : Number of non-augmented images per class
        include_misclassified : Whether to include misclassified images
        image_size            : Image size for model input
    """

    # ---------------------------
    # Setup
    # ---------------------------
    os.makedirs(os.path.join(output_folder, "original"), exist_ok=True)
    os.makedirs(os.path.join(output_folder, "gradcam"), exist_ok=True)
    gradcam = GradcamPlusPlus(model, model_modifier=ReplaceToLinear(), clone=True)

    all_images = []
    for label in labels:
        class_dir = os.path.join(dataset_path, label)
        if not os.path.isdir(class_dir):
            print(f"[WARN] Missing class folder: {class_dir}")
            continue

        images = [f for f in os.listdir(class_dir)
                  if not f.lower().startswith("aug") and f.lower().endswith((".jpg", ".jpeg", ".png"))]

        if len(images) == 0:
            print(f"[WARN] No non-aug images found in {class_dir}")
            continue

        # Randomly select limited number of images per class
        selected_imgs = random.sample(images, min(samples_per_class, len(images)))

        for img_name in selected_imgs:
            all_images.append((os.path.join(class_dir, img_name), label))

    print(f"\nTotal selected images: {len(all_images)}\n")

    y_true, y_pred = [], []
    csv_rows = []

    # ---------------------------
    # Process Images
    # ---------------------------
    for idx, (img_path, true_label) in enumerate(tqdm(all_images, desc="Processing Images", unit="image"), 1):
        img = tf.keras.utils.load_img(img_path, target_size=image_size)
        img_array = np.array(img)  # Convert to numpy array first
        img_array = preprocess_image(img_array)  # Apply your custom preprocessing
        input_tensor = np.expand_dims(img_array, axis=0)  # Add batch dimension

        preds = model.predict(input_tensor, verbose=0)[0]
        pred_idx = np.argmax(preds)
        pred_label = labels[pred_idx]
        conf = float(preds[pred_idx])

        y_true.append(labels.index(true_label))
        y_pred.append(pred_idx)

        correct = (pred_label == true_label)

        # Skip if not correct (unless include_misclassified=True)
        if not correct and not include_misclassified:
            print(f"[SKIP] {os.path.basename(img_path)} | True={true_label} | Pred={pred_label}")
            continue

        # Grad-CAM++
        cam = gradcam(CategoricalScore([pred_idx]), input_tensor, penultimate_layer=target_layer_name)[0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        # Overlay
        heatmap = plt.cm.jet(cam)[..., :3]
        overlay = 0.25 * heatmap + 0.75 * img_array
        overlay = np.uint8(255 * np.clip(overlay, 0, 1))

        # Save original & Grad-CAM++
        base_name = os.path.basename(img_path)
        orig_save = os.path.join(output_folder, "original", base_name)
        gradcam_save = os.path.join(output_folder, "gradcam", base_name)

        tf.keras.utils.save_img(orig_save, img_array)
        tf.keras.utils.save_img(gradcam_save, overlay)

        csv_rows.append([
            base_name,
            true_label,
            pred_label,
            round(conf, 5),
            "Correct" if correct else "Incorrect"
        ])

        #print(f"[{idx}/{len(all_images)}] {base_name} | True={true_label} | Pred={pred_label} | Conf={conf:.3f} | {'✅' if correct else '❌'}")

    # ---------------------------
    # Save CSV
    # ---------------------------
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image_name", "true_class", "pred_class", "confidence", "status"])
        writer.writerows(csv_rows)

    # ---------------------------
    # Compute Metrics
    # ---------------------------
    if len(y_true) > 0:
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        print("\n--- Final Metrics ---")
        print(f"Accuracy : {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall   : {rec:.4f}")
        print(f"F1 Score : {f1:.4f}")
    else:
        print("\n⚠️ No samples processed (check filters or include_misclassified flag).")

    print(f"\n✅ Original & Grad-CAM++ images saved in '{output_folder}'")
    print(f"✅ CSV saved at '{csv_path}'")

    return



In [ ]:

# ---------------------------
# Example usage
# ---------------------------
model = tf.keras.models.load_model("Proposed CBAM-Xception-DermNet")

generate_gradcam_pp_dataset(
    model=model,
    dataset_path="dataset_folder",
    labels=['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'],
    target_layer_name="block14_sepconv2_act",
    output_folder="gradcam_output",
    csv_path="gradcam_results.csv",
    samples_per_class=50,
    include_misclassified=False,
    image_size=(224, 224)
)


In [2]:
import pandas as pd
ham10000_map = {
    'akiec': 'Actinic keratoses and intraepithelial carcinoma',
    'bcc': 'Basal cell carcinoma',
    'bkl': 'Benign keratosis-like lesions',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic nevi',
    'vasc': 'Vascular lesions'
}
CSV_PATH = "Proposed_Dataset_explanations.csv"
df = pd.read_csv(CSV_PATH)
df["predicted_class"] = df["predicted_class"].map(ham10000_map)

#update it to save in working dir
CSV_PATH = "Proposed_Final_Dataset.csv"
df.to_csv(CSV_PATH, index=False)

print("Mapping complete! File updated in place:", CSV_PATH)
print(df.head())

Mapping complete! File updated in place: Proposed_Final_Dataset.csv
                              original_image  \
0  gradcam_dataset/original\ISIC_0024330.jpg   
1  gradcam_dataset/original\ISIC_0024370.jpg   
2  gradcam_dataset/original\ISIC_0024381.jpg   
3  gradcam_dataset/original\ISIC_0024388.jpg   
4  gradcam_dataset/original\ISIC_0024396.jpg   

                              gradcam_image                predicted_class  \
0  gradcam_dataset/gradcam\ISIC_0024330.jpg                 Dermatofibroma   
1  gradcam_dataset/gradcam\ISIC_0024370.jpg               Vascular lesions   
2  gradcam_dataset/gradcam\ISIC_0024381.jpg  Benign keratosis-like lesions   
3  gradcam_dataset/gradcam\ISIC_0024388.jpg               Melanocytic nevi   
4  gradcam_dataset/gradcam\ISIC_0024396.jpg                 Dermatofibroma   

                                 gradcam_description  
0  The model focused on the lesion's central pink...  
1  The model focused on the lesion's central pale...  
2  The mo